# 📱 OlegShifter Android APK Builder
## Google Colab Edition

Этот ноутбук автоматически собирает полноценное Android-приложение для подключения к SOCKS5-прокси OlegShifter.

**Преимущества:**
- ✅ Полностью автоматизированная сборка
- ✅ Не требует Linux/WSL на вашем компьютере
- ✅ Работает в Google Colab (бесплатно!)
- ✅ Результат готов к установке на Android

**Время сборки:** 10-15 минут

## 🔧 Шаг 1: Установка зависимостей

In [ ]:
import os
import sys

print("📦 Установка системных зависимостей...")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq build-essential git python3-dev python3-pip openjdk-17-jdk unzip wget > /dev/null 2>&1

print("📦 Установка Python зависимостей...")
!pip install -q buildozer cython kivy cryptography websockets protobuf pyjnius

print("\n✅ Все зависимости установлены!")

## 📥 Шаг 2: Загрузка Android NDK и SDK

In [ ]:
import os
import subprocess

ANDROID_HOME = os.path.expanduser('~/android')
os.makedirs(ANDROID_HOME, exist_ok=True)
os.chdir(ANDROID_HOME)

print(f"📍 Рабочая директория: {ANDROID_HOME}")

# Загрузка NDK
NDK_PATH = os.path.join(ANDROID_HOME, 'android-ndk-r25b')
if not os.path.exists(NDK_PATH):
    print("\n📥 Загружаю Android NDK r25b (~ 600 MB)...")
    !wget -q https://dl.google.com/android/repository/android-ndk-r25b-linux-x86_64.zip -O ndk.zip
    print("   Распаковываю...")
    !unzip -q ndk.zip
    !rm ndk.zip
    print("   ✅ NDK готов")
else:
    print(f"   ✓ NDK уже установлен: {NDK_PATH}")

# Загрузка SDK
SDK_PATH = os.path.join(ANDROID_HOME, 'android-sdk')
if not os.path.exists(SDK_PATH):
    print("\n📥 Загружаю Android SDK (~ 200 MB)...")
    !wget -q https://dl.google.com/android/repository/commandlinetools-linux-9477386_latest.zip -O sdk.zip
    os.makedirs(os.path.join(SDK_PATH, 'cmdline-tools'), exist_ok=True)
    print("   Распаковываю...")
    !unzip -q sdk.zip -d {SDK_PATH}/cmdline-tools/
    !mv {SDK_PATH}/cmdline-tools/cmdline-tools {SDK_PATH}/cmdline-tools/latest
    !rm sdk.zip
    
    print("   Установка пакетов SDK...")
    os.environ['ANDROID_HOME'] = SDK_PATH
    !yes | {SDK_PATH}/cmdline-tools/latest/bin/sdkmanager "platforms;android-31" "build-tools;33.0.0" 2>&1 | grep -E "(Installed|Updating|Complete)"
    print("   ✅ SDK готов")
else:
    print(f"   ✓ SDK уже установлен: {SDK_PATH}")

print("\n✅ Android инструменты готовы к работе!")

## 📂 Шаг 3: Подготовка проекта

In [ ]:
import os

# Вариант 1: Клонирование из GitHub
# ⬇️ ОТРЕДАКТИРУЙТЕ URL ВАШ РЕПОЗИТОРИЙ:
GITHUB_URL = "https://github.com/youruser/OlegShifter.git"
# GITHUB_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # Измените это!

PROJECT_PATH = '/content/OlegShifter'
ANDROID_CLIENT_PATH = os.path.join(PROJECT_PATH, 'android_kivy_client')

print("📂 Подготовка проекта...\n")

if not os.path.exists(PROJECT_PATH):
    print(f"📥 Клонирую репозиторий: {GITHUB_URL}")
    os.chdir('/content')
    result = os.system(f'git clone {GITHUB_URL} 2>&1 | tail -3')
    if result != 0:
        print("\n⚠️  ОШИБКА: Не удалось клонировать репозиторий")
        print("Проверьте URL и интернет-соединение")
        raise Exception("Git clone failed")
else:
    print(f"✓ Проект уже клонирован в {PROJECT_PATH}")

os.chdir(ANDROID_CLIENT_PATH)
print(f"\n✅ Проект готов: {ANDROID_CLIENT_PATH}")
print(f"   Содержимое:")
!ls -lah | head -15

## 🔨 Шаг 4: Сборка APK (это займёт 10-15 минут)

In [ ]:
import os
import time

ANDROID_HOME = os.path.expanduser('~/android')
os.environ['ANDROID_SDK_ROOT'] = os.path.join(ANDROID_HOME, 'android-sdk')
os.environ['ANDROID_NDK_HOME'] = os.path.join(ANDROID_HOME, 'android-ndk-r25b')
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

ANDROID_CLIENT_PATH = '/content/OlegShifter/android_kivy_client'
os.chdir(ANDROID_CLIENT_PATH)

print("🔨 НАЧАЛО СБОРКИ APK")
print("=" * 50)
print(f"Android SDK: {os.environ['ANDROID_SDK_ROOT']}")
print(f"Android NDK: {os.environ['ANDROID_NDK_HOME']}")
print(f"Java Home: {os.environ['JAVA_HOME']}")
print("=" * 50)
print("⏱️  Это займёт 10-15 минут...\n")

start_time = time.time()

# Запускаем buildozer
result = os.system('buildozer -v android debug 2>&1 | tail -50')

elapsed = time.time() - start_time
minutes = int(elapsed // 60)
seconds = int(elapsed % 60)

print("\n" + "=" * 50)
if result == 0:
    print(f"✅✅✅ УСПЕХ! APK СОБРАН ✅✅✅")
    print(f"   Время сборки: {minutes}m {seconds}s")
else:
    print(f"❌ Сборка завершилась с ошибкой (код {result})")
    print(f"   Время сборки: {minutes}m {seconds}s")
print("=" * 50)

## 📥 Шаг 5: Загрузка APK на компьютер

In [ ]:
import os
from google.colab import files

ANDROID_CLIENT_PATH = '/content/OlegShifter/android_kivy_client'
os.chdir(ANDROID_CLIENT_PATH)

# Поиск APK файла
import glob

apk_files = glob.glob('bin/*.apk')

if apk_files:
    apk_path = apk_files[0]
    apk_size = os.path.getsize(apk_path) / (1024 * 1024)  # MB
    print(f"📦 Найден APK: {os.path.basename(apk_path)}")
    print(f"   Размер: {apk_size:.1f} MB")
    print(f"\n📥 Загружаю на компьютер...")
    files.download(apk_path)
    print(f"\n✅ APK загружен!")
    print(f"\n📋 Дальше установите на Android:")
    print(f"   1. Скачайте APK (должен быть в Downloads)")
    print(f"   2. Подключите Android-устройство (USB)")
    print(f"   3. В PowerShell/Terminal: adb install {os.path.basename(apk_path)}")
else:
    print("❌ APK не найден!")
    print(f"\nПроверьте содержимое bin/: ")
    !ls -la bin/ 2>/dev/null || echo "   Папка bin/ не существует"

## 📱 Шаг 6: Установка на Android

### Вариант 1: Через ADB (рекомендуется)

**На Windows (PowerShell):**
```powershell
# 1. Установите ADB (если ещё нет)
winget install Google.AndroidStudio

# 2. Включите USB Debug на телефоне
#    Settings → Developer Options → USB Debugging

# 3. Подключите телефон и выполните:
adb install olegshifter-1.0.0-debug.apk
```

**На Linux/Mac:**
```bash
sudo apt install android-tools-adb  # Linux
# или
brew install android-platform-tools  # Mac

adb install olegshifter-1.0.0-debug.apk
```

### Вариант 2: Прямая установка на телефон

1. Скачайте APK на компьютер (кнопка выше)
2. Отправьте файл на телефон (Telegram, email, облако)
3. На телефоне откройте файл менеджер
4. Найдите APK и нажмите на него
5. Подтвердите установку

### Вариант 3: Через Telegram/Email

```bash
# Загрузите APK в Telegram/email самому себе
# Затем скачайте на телефоне и установите
```

## ⚙️ Использование приложения

После установки:

1. **Откройте OlegShifter**
2. **Введите параметры сервера:**
   - Host: IP/домен прокси-сервера
   - Port: Порт сервера (по умолчанию 8080)
   - Channels: Количество каналов (1-10)
   - SOCKS5 Port: Локальный SOCKS5 порт (1080)
   - Preshared Secret: Ключ (если нужен)
   - Use TLS: Включить TLS (если сервер требует)
3. **Нажмите "Подключиться"**
4. **Настройте другие приложения на прокси:**
   - localhost:1080 (SOCKS5)

## 🐛 Решение проблем

**Приложение не подключается:**
- Проверьте, запущен ли сервер
- Убедитесь, что хост и порт правильные
- Посмотрите логи в приложении (вкладка "Логи")

**APK не устанавливается:**
- На телефоне разрешите установку из неизвестных источников
- Settings → Security → Unknown sources → ✓

**Проблемы с ADB:**
- Включите USB Debug на телефоне
- Выполните: `adb kill-server && adb start-server`

---

**Вопросы или проблемы?** Откройте Issue на GitHub!